In [62]:
import numpy as np
import pandas as pd

In [63]:
df = pd.read_csv('/content/drive/MyDrive/Deep Learning/diabetes (1).csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [64]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [65]:
x = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [66]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
x = sc.fit_transform(x)

In [67]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)

In [114]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout

In [69]:
model = Sequential()

model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [70]:
model.fit(x_train, y_train, batch_size=32, epochs=100, validation_data=(x_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4430 - loss: 0.7280 - val_accuracy: 0.6169 - val_loss: 0.6754
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6287 - loss: 0.6648 - val_accuracy: 0.7013 - val_loss: 0.6247
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6971 - loss: 0.6197 - val_accuracy: 0.7857 - val_loss: 0.5853
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7329 - loss: 0.5849 - val_accuracy: 0.7662 - val_loss: 0.5558
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7410 - loss: 0.5581 - val_accuracy: 0.7922 - val_loss: 0.5301
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7508 - loss: 0.5364 - val_accuracy: 0.7987 - val_loss: 0.5092
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7606 - loss: 0.5196 - val_accuracy: 0.7987 - val_loss: 0.4928
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7704 - loss: 0.5059 - val_accuracy: 0.7987 - 

In [71]:
# 1. how to select appropriate optimizer
# 2. No. of nodes in a layer
# 3. How to select no. of layers
# 4. All in one model

In [72]:
import kerastuner as kt

In [73]:
def build_model(hp):

  model = Sequential()


  model.add(Dense(32, activation='relu', input_dim=8))
  model.add(Dense(1, activation='sigmoid'))

  optimizer = hp.Choice('optimizer', values = ['adam', 'rmsprop', 'adagrad', 'adadelta'])

  model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [74]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [75]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test), verbose=1)

In [76]:
tuner.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 3 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7597402334213257

Trial 1 summary
Hyperparameters:
optimizer: adam
Score: 0.7467532753944397

Trial 0 summary
Hyperparameters:
optimizer: adadelta
Score: 0.6428571343421936

Trial 2 summary
Hyperparameters:
optimizer: adagrad
Score: 0.5324675440788269


In [77]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [78]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [79]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [80]:
model.fit(x_train, y_train, batch_size=32, epochs=100, initial_epoch= 6,validation_data=(x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.7117 - loss: 0.5437 - val_accuracy: 0.7532 - val_loss: 0.5169
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7215 - loss: 0.5232 - val_accuracy: 0.7662 - val_loss: 0.5049
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7264 - loss: 0.5102 - val_accuracy: 0.7597 - val_loss: 0.4943
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7427 - loss: 0.4997 - val_accuracy: 0.7532 - val_loss: 0.4866
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7557 - loss: 0.4901 - val_accuracy: 0.7597 - val_loss: 0.4815
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7606 - loss: 0.4832 - val_accuracy: 0.7727 - val_loss: 0.4782
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7638 - loss: 0.4774 - val_accuracy: 0.7727 - val_loss: 0.4756
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7704 - loss: 0.4723 - val_accuracy: 0.7

In [81]:
def build_model(hp):
  model = Sequential()

  units = hp.Int('units', 8, 128, step=8)

  model.add(Dense(units = units, activation='relu', input_dim=8))
  model.add(Dense(1, activation = 'sigmoid'))

  model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [82]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    directory='mydir',
    project_name='myproject'
)

Reloading Tuner from mydir/myproject/tuner0.json


In [83]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test), verbose=1)

In [84]:
tuner.get_best_hyperparameters()[0].values

{'units': 128}

In [85]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [86]:
model.fit(x_train, y_train, epochs=100, initial_epoch = 5, validation_data=(x_test, y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7687 - loss: 0.4827 - val_accuracy: 0.8052 - val_loss: 0.4833
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7720 - loss: 0.4672 - val_accuracy: 0.8052 - val_loss: 0.4778
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7752 - loss: 0.4605 - val_accuracy: 0.7922 - val_loss: 0.4753
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7801 - loss: 0.4565 - val_accuracy: 0.7987 - val_loss: 0.4745
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7818 - loss: 0.4531 - val_accuracy: 0.7922 - val_loss: 0.4744
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7769 - loss: 0.4507 - val_accuracy: 0.7987 - val_loss: 0.4749
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7736 - loss: 0.4479 - val_accuracy: 0.7857 - val_loss: 0.4760
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7801 - loss: 0.4456 - val_accuracy: 0.792

In [87]:
def build_model(hp):
  model = Sequential()

  model.add(Dense(72, activation='relu', input_dim=8))

  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

    model.add(Dense(72, activation='relu'))

  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [88]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,
    directory='mydir',
    project_name='myproject1'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [89]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test), verbose=1)

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.8051947951316833

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 12s


In [90]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 5}

In [92]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [93]:
model.fit(x_train, y_train, epochs=100, initial_epoch = 5, validation_data=(x_test, y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7704 - loss: 0.4655 - val_accuracy: 0.7857 - val_loss: 0.4789
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7752 - loss: 0.4417 - val_accuracy: 0.7857 - val_loss: 0.4864
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7866 - loss: 0.4318 - val_accuracy: 0.7792 - val_loss: 0.4811
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7883 - loss: 0.4260 - val_accuracy: 0.7922 - val_loss: 0.4815
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8111 - loss: 0.4134 - val_accuracy: 0.7403 - val_loss: 0.5446
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7997 - loss: 0.4128 - val_accuracy: 0.7857 - val_loss: 0.4920
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8078 - loss: 0.4017 - val_accuracy: 0.7987 - val_loss: 0.4902
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8094 - loss: 0.3987 - val_accuracy: 0.811

In [115]:
def build_model(hp):

  model = Sequential()

  counter = 0

  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

    if counter == 0:

      model.add(
          Dense(
              hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
              activation = hp.Choice('activation'+str(i), values=['relu', 'tanh', 'sigmoid', 'selu']),
              input_dim=8
          )
      )
      model.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7])))

    else:

      model.add(
          Dense(
              hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
              activation = hp.Choice('activation'+str(i), values=['relu', 'tanh', 'sigmoid', 'selu'])
          )
      )
      model.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7])))
    counter += 1

  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer=hp.Choice('optimizer', values=['adam', 'rmsprop', 'adagrad', 'sgd']), loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [116]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,
    directory='mydir',
    project_name='myproject2'
)

Reloading Tuner from mydir/myproject2/tuner0.json


In [117]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test), verbose=1)

In [118]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3,
 'units0': 112,
 'activation0': 'sigmoid',
 'optimizer': 'adagrad',
 'units1': 96,
 'activation1': 'tanh',
 'units2': 48,
 'activation2': 'relu',
 'units3': 64,
 'activation3': 'selu',
 'units4': 24,
 'activation4': 'selu',
 'units5': 88,
 'activation5': 'tanh',
 'units6': 56,
 'activation6': 'relu',
 'units7': 48,
 'activation7': 'selu'}

In [119]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adagrad', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [120]:
model.fit(x_train, y_train, epochs=200, initial_epoch=5, validation_data=(x_test, y_test))

Epoch 6/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.5814 - loss: 0.6757 - val_accuracy: 0.6494 - val_loss: 0.6543
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6401 - loss: 0.6617 - val_accuracy: 0.6494 - val_loss: 0.6492
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6401 - loss: 0.6558 - val_accuracy: 0.6494 - val_loss: 0.6465
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6466 - loss: 0.6541 - val_accuracy: 0.6429 - val_loss: 0.6451
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6531 - loss: 0.6474 - val_accuracy: 0.6429 - val_loss: 0.6441
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6547 - loss: 0.6411 - val_accuracy: 0.6429 - val_loss: 0.6431
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6482 - loss: 0.6488 - val_accuracy: 0.6429 - val_loss: 0.6423
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6515 - loss: 0.6488 - val_accuracy: 0.64